In [9]:
import pandas as pd
import re
import math
import json
import requests
from bs4 import BeautifulSoup
from io import StringIO
from variables import TOUR_NAME, POKEDATA_CSV, DAY1_ROUNDS, CATEGORY

In [10]:
def clean_name(input_string):
    result = re.sub(r'\s*\[.*?\]\s*', '', input_string)
    result = re.sub(r'STATIC SEATING \(\d+\)\s*', '', result)
    result = re.sub(r'>.*?>', '', result)
    result = re.sub(r'>TABLE \d+ ', '', result)
    return result

In [11]:
pairings_df = pd.read_csv(StringIO(requests.get(POKEDATA_CSV).content.decode('utf-8')), sep='\t', header=None, encoding='utf-8')
pairings_df.rename(columns={0:'Player',1:'Opponent',2:'Result',3:'Points',4:'Round'}, inplace=True)
pairings_df['Player'] = pairings_df['Player'].apply(clean_name)
pairings_df['Opponent'] = pairings_df['Opponent'].apply(clean_name)
pairings_df = pairings_df[(pairings_df['Opponent'] != 'BYE') & (pairings_df['Opponent'] != 'LATE')]

In [12]:
deck_df = pd.read_excel(f'standings/{TOUR_NAME}_standings.xlsx', sheet_name=CATEGORY)
deck_df['Placement'] = deck_df['Placement'].apply(lambda x: "Top {}".format(pow(2, math.ceil(math.log(x, 2)))))
deck_df['Day 2'] = deck_df['Player'].map(lambda player: pairings_df.groupby('Player')['Round'].count().get(player, 0) > DAY1_ROUNDS)


In [13]:
# Check missing players
player_index = 0
for player in pairings_df['Player'].unique():
    if player not in deck_df['Player'].unique():
        print(player_index, player)
    player_index+=1

0 Ryuki Okada
179 Emilia Miele
331 Princess Basser
1290 Bradley M. Smith
1466 Alex Alvarez
1965 Enzo Rudy SEKKAÏ
2526 Van Jones
3006 Maximilien Mairey
3088 Jesus Nogales
3347 Maxence de Lussac
3348 Robert-Jan Bos
3349 Thomas Asare
3350 Junior Chala
3351 Alexander Mehta
3352 Jack Cheshire
3353 Zhen Dong


In [14]:
deck_dict = deck_df.set_index('Player')['Deck'].to_dict()

In [15]:
matchups_df = pd.DataFrame(columns=['Deck', 'Opposing Deck', 'Wins', 'Losses', 'Ties'])
# pairings_final_df = pd.DataFrame(columns=['Player','Opponent','Result','Points','Round'])

for index, row in pairings_df.iterrows():
    try:
        player_deck = deck_dict[row['Player']]
        opp_deck = deck_dict[row['Opponent']]
        matchup = matchups_df.loc[(matchups_df['Deck'] == player_deck) & (matchups_df['Opposing Deck'] == opp_deck)]
        if row['Round'] == DAY1_ROUNDS and row['Points'] == ((DAY1_ROUNDS-3)*3 + 1) and row['Result'] == 'T':
            # print(row)
            continue
        if row['Result'] == 'T':
            if len(matchup) == 0:
                matchups_df.loc[len(matchups_df)] = player_deck, opp_deck, 0, 0, 1
            else:
                matchups_df.loc[matchup.index, 'Ties'] += 1
        elif row['Result'] == 'W':
            if len(matchup) == 0:
                matchups_df.loc[len(matchups_df)] = player_deck, opp_deck, 1, 0, 0
            else:
                matchups_df.loc[matchup.index, 'Wins'] += 1
        elif row['Result'] == 'L':
            if len(matchup) == 0:
                    matchups_df.loc[len(matchups_df)] = player_deck, opp_deck, 0, 1, 0
            else:
                matchups_df.loc[matchup.index, 'Losses'] += 1
        # pairings_final_df.loc[len(pairings_final_df)] = row
    except:
        continue


In [ ]:
with pd.ExcelWriter(f'datasets/{TOUR_NAME}_{CATEGORY}.xlsx') as writer:
    # pairings_final_df.to_excel(writer, sheet_name='pairings', index=False)
    deck_df.to_excel(writer, sheet_name='decks', index=False)
    matchups_df.to_excel(writer, sheet_name='matchups', index=False)